# Baseline LDM — perio-KPT (Google Colab)

**Run the next code cell first** (`Setup + imports`). Without it you get `No module named 'src'`.

Colab in VS Code often starts in `/content` while your repo is elsewhere — Setup searches for the folder that contains `src/`.

## Setup + imports (run this cell first)

In [ ]:
!git clone https://github.com/Rafa13io/computer_vision_project.git

In [ ]:
import os
import sys
from pathlib import Path


def find_project_root() -> Path:
    """Locate repo root (directory containing src/utils/config.py)."""
    checked: set[Path] = set()
    candidates: list[Path] = []

    # Current dir and parents (notebook may live in notebooks/)
    p = Path.cwd().resolve()
    for _ in range(8):
        candidates.append(p)
        if p.parent == p:
            break
        p = p.parent

    # Common Colab locations
    candidates.extend(
        [
            Path("/content/computer_vision_project"),
            Path("/content/drive/MyDrive/computer_vision_project"),
            Path("/content/drive/MyDrive/Colab Notebooks/computer_vision_project"),
        ]
    )

    for cand in candidates:
        cand = cand.resolve()
        if cand in checked:
            continue
        checked.add(cand)
        if (cand / "src" / "utils" / "config.py").is_file():
            return cand

    raise FileNotFoundError(
        "Project root not found. On Colab, clone or upload the full repo, e.g.\n"
        "  !git clone https://github.com/Rafa13io/computer_vision_project.git\n"
        "Then re-run this cell."
    )


ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("Working directory:", Path.cwd())
print("src on path:", any("computer_vision" in x or x.endswith("/content") or "src" in x for x in sys.path[:3]))

# Dependencies
import subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")],
    check=False,
)

import torch
from src.utils.config import load_config, project_root
from src.utils.seed import set_seed

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

ModuleNotFoundError: No module named 'src'

## Globals

In [ ]:
CFG_PATH = ROOT / "configs/baseline_ldm.yaml"
cfg = load_config(CFG_PATH)
set_seed(int(cfg["seed"]))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Data

1. Download [perio-KPT](https://zenodo.org/records/17272200) → `data/perio-KPT/`
2. Run the cell below (uses `ROOT` from Setup — no separate `!python` needed)

## Train

```bash
!python -m src.train.train_vae --config configs/baseline_ldm.yaml
!python -m src.train.train_ldm --config configs/baseline_ldm.yaml
```

## Evaluation — samples & failure analysis

```bash
!python -m src.train.sample --config configs/baseline_ldm.yaml
```

Document structural errors: wrong bone level, CEJ blur, root shape, texture-only fakes.

# Build manifest (run after dataset is in data/perio-KPT/)
import subprocess

subprocess.check_call(
    [sys.executable, "-m", "src.data.build_manifest", "--config", "configs/baseline_ldm.yaml"],
    cwd=ROOT,
)



In [ ]:
!python -m src.data.build_manifest --config configs/baseline_ldm.yaml


/usr/bin/python3: Error while finding module specification for 'src.data.build_manifest' (ModuleNotFoundError: No module named 'src')


In [ ]:
!python -m src.train.train_vae --config configs/baseline_ldm.yaml


In [ ]:
!python -m src.train.train_ldm --config configs/baseline_ldm.yaml


In [ ]:
!python -m src.train.sample --config configs/baseline_ldm.yaml